# First step in co-reg
- Load files
    - HCR processed data (first round)
        - Manually check if the first round data was OK.
    - Cortical z-stack data
        - The last session with desired resolution
            - 400x400 for now. 
            - Try 700x700 later.
- Process files

    0. Attach data assets
    1. Swap z-stack registration tiff file dimension
    2. Save z-stack segmentation outline file (for QCing automatic landmarks)
    3. Save z-stack centroid (for generating automatic landmarks)
    4. Save filepaths


In [22]:
from pathlib import Path
import tifffile as tiff
import numpy as np
import pandas as pd
import cv2
import json
from scipy import ndimage as ndi

from lamf_analysis.code_ocean import code_ocean_utils as cou
from lamf_analysis.ophys import zstack_utils

from codeocean.data_asset import DataAssetSearchParams
from codeocean.components import SearchFilter

DATA_DIR = Path('/root/capsule/data/')

%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [23]:
subject_id = 767018
czstack_xy_size = 400 # either 400 or 700 (um in one dimension)

In [10]:
''' Now save_dir definition comes after attaching zstack_reg_dir (11/18/2025 JK)'''
# # # TODO: define save_dir based on the czstack data name
# # save_dir = Path('/root/capsule/scratch/755252_2024-12-19_coreg_cpsam/')
# # save_dir.mkdir(parents=True, exist_ok=True)
# # czstack_reg_path = '/root/capsule/data/multiplane-ophys_755252_2024-12-19_11-34-11_cortical-zstack-registration_2025-11-11_19-29-43/cortical_zstack_0/channel_0_ref_0/cortical_zstack_0_2xREG.tif'
# # czstack_seg_path = '/root/capsule/data/multiplane-ophys_755252_2024-12-19_11-34-11_cortical-zstack-segmentation_2025-11-14_00-34-06/channel_0_ref_0/segmentation_masks.tif'
# # hcr_centroid_path = '/root/capsule/data/HCR_755252_2025-07-02_13-00-00_processed_2025-07-18_09-27-44/cell_body_segmentation/cell_centroids.npy'

' Now save_dir definition comes after attaching zstack_reg_dir (11/18/2025 JK)'

# 0. Attach data assets

## HCR data - Round 1 (first data)

In [24]:
offset = 0
limit = 1000
data_asset_filters = [
    SearchFilter(
        key="tags",
        value="HCR"
    ),
    SearchFilter(
        key="tags",
        value="processed"
    ),
    SearchFilter(
        key="name",
        value=f"HCR_{subject_id}"
    ),
]
data_asset_params = DataAssetSearchParams(
        offset=offset,
        limit=limit,
        sort_order="desc",
        sort_field="name",
        archived=False,
        favorite=False,
        # query="name:'multiplane-ophys'",
        filters=data_asset_filters
    )

co_client = cou.get_co_client()
search_results = co_client.data_assets.search_data_assets(data_asset_params)

hcr_data_df = pd.DataFrame([{"date": r.name.split('_')[2], "id": r.id, "asset_name": r.name} for r in search_results.results]).sort_values('date')
print(hcr_data_df)
row_to_attach = hcr_data_df.iloc[0]
hcr_path = DATA_DIR / row_to_attach.asset_name
print(f'Attaching data from {row_to_attach.date}, id: {row_to_attach.id}, to {hcr_path}')
cou.attach_assets([row_to_attach.id])

         date                                    id  \
3  2025-08-20  179763ab-f225-4467-9b82-e05b34f344e2   
2  2025-09-05  6740caaa-e064-4000-aaf0-e840ab92718f   
1  2025-09-11  6a7bd6c7-8fd3-4188-8240-25ca770b82d5   
0  2025-09-18  cf6d4e49-d73f-4fb3-aa53-d8ba670adf7e   

                                          asset_name  
3  HCR_767018_2025-08-20_14-00-00_processed_2025-...  
2  HCR_767018_2025-09-05_13-00-00_processed_2025-...  
1  HCR_767018_2025-09-11_13-00-00_processed_2025-...  
0  HCR_767018_2025-09-18_13-00-00_processed_2025-...  
Attaching data from 2025-08-20, id: 179763ab-f225-4467-9b82-e05b34f344e2, to /root/capsule/data/HCR_767018_2025-08-20_14-00-00_processed_2025-09-10_00-33-24
asset_id: 179763ab-f225-4467-9b82-e05b34f344e2 - mount_state: unchanged


In [25]:
hcr_centroid_path = hcr_path / 'cell_body_segmentation/cell_centroids.npy'
assert hcr_centroid_path.exists()
fused_json_file = next(hcr_path.glob('tile_subset_corrected_ng.json'))

## Cortical z-stack - The last one with desired resolution
- In case of duplication (for testing, temporary), choose the latest processed file
- TODO: update zstack_utils to have processed dates and data provenance (for zstack segmentation)

### czstack registration

In [26]:
czstack_reg_df = zstack_utils.get_cortical_zstack_reg_df(subject_id)
print(czstack_reg_df)
zstack_reg_data_row = czstack_reg_df.query('xy_size_um == @czstack_xy_size').sort_values(['derived_asset_name', 'session_name']).iloc[-1]
zstack_reg_dir = DATA_DIR / zstack_reg_data_row.derived_asset_name
zstack_reg_asset_id = zstack_reg_data_row.derived_asset_id
print(f'Attaching data from {zstack_reg_data_row.session_name} to {zstack_reg_dir}')
cou.attach_assets([zstack_reg_asset_id])

                       derived_asset_id  \
0  5314a682-99bd-4730-887b-3dfd5d3c6db7   
1  1ed7883b-ff20-4849-9f5d-614123016299   
2  5dff5f8d-706e-40e7-80ce-944034d00c63   
3  57d745de-0df6-4bed-ba36-324c27a03ad0   
4  de9f15be-7a40-4d69-bb64-8dc304e4877d   
5  ea7201a4-3ff3-4cb7-b957-2dca37a042e6   
6  9f4388ad-5420-49c7-abc1-0a3e0bb8b74b   
7  c8bcec2f-6f4b-45ce-a76e-4046fa30454a   

                                  derived_asset_name  \
0  multiplane-ophys_767018_2025-02-19_14-02-36_co...   
1  multiplane-ophys_767018_2025-02-19_14-02-36_co...   
2  multiplane-ophys_767018_2025-02-18_12-28-32_co...   
3  multiplane-ophys_767018_2025-02-18_12-28-32_co...   
4  multiplane-ophys_767018_2025-02-13_14-09-49_co...   
5  multiplane-ophys_767018_2025-02-13_14-09-49_co...   
6  multiplane-ophys_767018_2025-01-22_14-04-11_co...   
7  multiplane-ophys_767018_2025-01-22_14-04-11_co...   

                                raw_asset_name       session_name  \
0  multiplane-ophys_767018_2025-02-19_

In [27]:
czstack_reg_path = zstack_reg_dir / 'cortical_zstack_0/channel_0_ref_0/cortical_zstack_0_2xREG.tif'
assert czstack_reg_path.exists()

## Assign and make save_dir

In [28]:
coreg_name = '_'.join(zstack_reg_dir.name.split('_')[1:3])
save_dir = Path(f'/root/capsule/scratch/{coreg_name}_coreg_cpsam/')
save_dir.mkdir(parents=True, exist_ok=True)

### czstack segmentation
- TODO: test data provenance from CodeOcean

In [29]:
process_name = 'cortical-zstack-segmentation'
seg_assets = cou.get_derived_assets(subject_id, process_name)
derived_asset_rows = []
for asset in seg_assets:
    if zstack_reg_asset_id in asset.provenance.data_assets:
        name = asset.name
        raw_asset_name = name.split(process_name)[0].rstrip('_')
        session_name = '_'.join(raw_asset_name.split('_')[1:3])
        if str(subject_id) != session_name.split('_')[0]:
            raise ValueError(f"Subject ID mismatch in asset '{name}': expected {subject_id}, found {session_name.split('_')[0]}")
        derived_asset_rows.append({
            'derived_asset_id': asset.id,
            'derived_asset_name': name,
            # 'raw_asset_name': raw_asset_name,
            'session_name': session_name,
            'input_data_assets': asset.provenance.data_assets
        })
derived_asset_df = pd.DataFrame(derived_asset_rows,
                                columns=['derived_asset_id',
                                        'derived_asset_name',
                                        # 'raw_asset_name',
                                        'session_name',
                                        'input_data_assets',
                                        ])
assert len(derived_asset_rows) == 1
print(derived_asset_rows[0])
zstack_seg_dir = DATA_DIR / derived_asset_rows[0]['derived_asset_name']
print(f'Attaching segmentation data to {zstack_seg_dir}')
cou.attach_assets([derived_asset_rows[0]['derived_asset_id']])

{'derived_asset_id': 'e95ea178-142e-4f7f-ab3f-3d422cf3696f', 'derived_asset_name': 'multiplane-ophys_767018_2025-02-13_14-09-49_cortical-zstack-segmentation_2025-11-14_03-42-00', 'session_name': '767018_2025-02-13', 'input_data_assets': ['de9f15be-7a40-4d69-bb64-8dc304e4877d']}
Attaching segmentation data to /root/capsule/data/multiplane-ophys_767018_2025-02-13_14-09-49_cortical-zstack-segmentation_2025-11-14_03-42-00
asset_id: e95ea178-142e-4f7f-ab3f-3d422cf3696f - mount_state: mounted


In [30]:
czstack_seg_path = zstack_seg_dir / 'channel_0_ref_0/segmentation_masks.tif'
assert czstack_seg_path.exists()

# 1. Swap z-stack registration tiff file dimension
- FIJI treats (450, 512, 512) as having 450 timepoints, instead of 450 z planes.
- Use ome tiff

In [31]:
zstack_reg_swapped_path = save_dir / f"{coreg_name}_reg-dim-swapped.ome.tif"

original_reg = tiff.imread(czstack_reg_path)
swapped = original_reg[np.newaxis, np.newaxis, ...]
tiff.imwrite(
    zstack_reg_swapped_path,
    swapped,
    ome=True,
    metadata={'axes': 'TCZYX'}
)

# 2. Save z-stack segmentation outline file (for QCing automatic landmarks)

In [32]:
czstack_seg_outline_path = save_dir / f'{coreg_name}_seg-mask-outline.tif'

czstack_masks = tiff.imread(czstack_seg_path)
czstack_masks_outline = np.zeros_like(czstack_masks,dtype=np.uint8)
kernel = np.ones((3, 3), np.uint8)
for i_plane, plane in enumerate(czstack_masks):
    ids = np.unique(plane)
    ids = ids[ids!=0]
    for id in ids:
        # Creates a uint8 array (1/0) that OpenCV can use
        mask = (plane==id).astype(np.uint8)
        eroded_mask = cv2.erode(mask, kernel, iterations=1)
        edge = cv2.subtract(mask, eroded_mask)==1
        czstack_masks_outline[i_plane][edge] += 1

tiff.imwrite(czstack_seg_outline_path, czstack_masks_outline)

# 3. Save z-stack centroid (for generating automatic landmarks)

In [33]:
czstack_centroid_path = save_dir / f'{coreg_name}_czstack_cell_centroids.csv'

czstack_masks = tiff.imread(czstack_seg_path)

czstack_cell_ids = np.unique(czstack_masks)
czstack_cell_ids = czstack_cell_ids[czstack_cell_ids!=0]

czstack_cell_centroids = ndi.center_of_mass(
    np.ones_like(czstack_masks),   # input
    labels=czstack_masks,          # label image
    index=czstack_cell_ids         # list of labels
)

czstack_cell_centroids = np.array(czstack_cell_centroids)
czstack_cell_centroids_df = pd.DataFrame()
czstack_cell_centroids_df['czstack_cell_id'] = czstack_cell_ids
czstack_cell_centroids_df['czstack_z'] = czstack_cell_centroids[:,0]
czstack_cell_centroids_df['czstack_y'] = czstack_cell_centroids[:,1]
czstack_cell_centroids_df['czstack_x'] = czstack_cell_centroids[:,2]
czstack_cell_centroids_df.set_index('czstack_cell_id', inplace=True)
czstack_cell_centroids_df.to_csv(czstack_centroid_path)

# 4. Save filepaths

In [34]:
save_path = save_dir / f'{coreg_name}_filepaths.json'

filepaths_dict = {
    'czstack_reg_path': str(czstack_reg_path),
    'czstack_reg_dim_swapped_path': str(zstack_reg_swapped_path),
    'czstack_seg_path': str(czstack_seg_path),
    'czstack_seg_outline_path': str(czstack_seg_outline_path),
    'czstack_centroid_path': str(czstack_centroid_path),
    'hcr_centroid_path': str(hcr_centroid_path),
    'fused_json_file': str(fused_json_file)
}

with open(save_path, 'w') as f:
    json.dump(filepaths_dict, f)

# Now open files in FIJI and find landmarks!